# 04 Baseline Summary

## [ 프로젝트 개요 ]

본 프로젝트는 한국미디어패널(KMP) 2020~2025 개인조사 데이터를 활용하여 통신사 이탈 가능성을 예측하는 것을 목표로 한다.
분석 대상은 두 가지 라벨로 구성하였다.

- `churn_any`: 전년 대비 이용 통신사가 변경되었는지 여부
- `churn_to_mvno`: 전년 메이저 통신사(1/2/3) 이용자가 다음 해 알뜰폰(MVNO=4)으로 이동했는지 여부

두 라벨은 모두 통신사 변경과 관련되어 있지만, 이벤트의 성격과 난이도는 다르다.
`churn_any`는 상대적으로 일반적인 통신사 이동을 의미하고, `churn_to_mvno`는 그중에서도 메이저 통신사에서 MVNO로 이동하는 희소한 사례만을 대상으로 한다.

---

## [ 데이터 및 분석 설정 ]

분석에는 KMP 2020~2025 개인조사 CSV(`p20 ~ p25`)와 코드북(`P_codebook_v32.xlsx`)을 사용하였다.
데이터는 `pid` 기준으로 연도 간 전환을 연결한 transition 기반 long panel 형태로 구성하였다.
한 행은 `(pid, year_t0=t-1 → year_t1=t)` 구조를 가지며, 예측에는 항상 `t-1` 시점 변수만 사용하였다.

이번 baseline 및 보강 실험에서는 다음 7개 핵심 변수를 입력 변수로 사용하였다.

- 스마트폰/5G 관련
- 음성 무제한 여부
- 데이터 무제한 여부
- 월평균 휴대폰 이용금액
- 통신 지출/가계 관련
- 특정 서비스 이용 여부
- 이용 행태 관련

또한 동일 인물의 정보가 train/test에 동시에 포함되지 않도록 `pid` 기준 그룹 분할을 적용하였다.

---

## [ 표시용 변수명 정리 ]

보고서와 시각화에서는 가독성을 위해 일부 코드형 변수명을 다음과 같이 해석하여 함께 사용하였다.

| 원본 컬럼명 | 표시용 이름 |
|---|---|
| `a03002_tminus1` | 스마트폰/5G 관련 |
| `a03024_tminus1` | 음성 무제한 여부 |
| `a03026_tminus1` | 데이터 무제한 여부 |
| `c01002_tminus1` | 월평균 휴대폰 이용금액 |
| `c01004_tminus1` | 통신 지출/가계 관련 |
| `c02003_tminus1` | 특정 서비스 이용 여부 |
| `c02001_tminus1` | 이용 행태 관련 |

이는 데이터셋 내부 컬럼명을 변경한 것이 아니라, 결과 해석과 시각화 단계에서만 표시용 이름을 함께 사용한 것이다.

---

## [ 비교 모델 구성 최신화 ]

초기 baseline에서는 `LogisticRegression`, `RandomForest`를 중심으로 비교하였다.
이후 보강 단계에서는 다음 모델을 추가하여 비교 폭을 넓혔다.

- `DecisionTree`
- `GradientBoosting`
- `XGBoost`

즉, 현재 프로젝트는 다음 다섯 모델을 비교하는 구조로 확장되었다.

| 모델 | 계열 |
|---|---|
| `LogisticRegression` | 선형 모델 |
| `DecisionTree` | 단일 트리 |
| `RandomForest` | 배깅 앙상블 |
| `GradientBoosting` | 부스팅 앙상블 |
| `XGBoost` | 실무형 부스팅 |

이를 통해 단순히 모델 수를 늘리는 데 그치지 않고,
현재 문제에서 어떤 모델 계열이 더 적합한지 구조적으로 비교할 수 있게 되었다.

---

## [ 라벨 분포 요약 ]

전처리 완료 후 전체 데이터는 41,299행으로 구성되었다.

- `churn_any` 양성 비율: 약 36.3%
- `churn_to_mvno` 양성 비율: 약 1.25%

즉, `churn_any`는 비교적 학습 가능한 수준의 분포를 가지는 반면,
`churn_to_mvno`는 매우 희소한 클래스이므로 예측 난도가 훨씬 높다.

---

## [ churn_any 결과 요약 ]

`churn_any` baseline 결과에서는 `RandomForest`와 `DecisionTree`가 Recall과 F1 기준 상대적으로 강한 성능을 보였다.

특히 `RandomForest`는 Recall `0.6192`, F1 `0.4832`를 기록하여 baseline 단계에서 가장 균형 잡힌 성능을 보여주었다.

`DecisionTree`도 Recall `0.5990`, F1 `0.4767`로 유사한 수준의 성능을 보였지만, 전반적인 안정성 측면에서는 `RandomForest`가 더 적절한 기준 모델로 해석되었다.

반면 `GradientBoosting`과 `XGBoost`는 Accuracy, ROC-AUC, PR-AUC는 비교적 높게 나타났으나, 실제 이탈자 예측을 매우 보수적으로 수행하여 Recall과 F1이 크게 낮았다.

따라서 `churn_any` 문제에서는 단순 정확도보다 실제 이탈자 탐지 성능을 중시할 때 `RandomForest`가 가장 실질적인 baseline 모델로 판단되었다.

---

## [ churn_to_mvno 결과 요약 ]

`churn_to_mvno`는 양성 비율이 약 `1.01%` 수준인 매우 희소한 클래스 문제이므로, Accuracy보다 Recall, PR-AUC, F1 중심 해석이 필요했다.

이번 baseline 비교에서는 `DecisionTree`와 `LogisticRegression`이 실제 양성 탐지 측면에서 상대적으로 더 실질적인 성능을 보였다.

`DecisionTree`는 Recall `0.5476`, F1 `0.0282`를 기록하였고, `LogisticRegression`도 Recall `0.5238`, F1 `0.0261`, ROC-AUC `0.6264`, PR-AUC `0.0182`로 비교적 안정적인 baseline 역할을 수행하였다.

반면 `RandomForest`는 실제 양성 탐지 성능이 더 낮았고, `GradientBoosting`과 `XGBoost`는 양성 예측을 거의 수행하지 못하였다.

즉 `churn_to_mvno` 문제에서는 복잡한 부스팅 모델보다 `LogisticRegression`과 `DecisionTree`가 더 실질적인 baseline 모델로 해석되었다.

---

## [ Baseline 요약 해석 ]

이번 baseline 비교에서는 `LogisticRegression`, `DecisionTree`, `RandomForest`, `GradientBoosting`, `XGBoost`를 함께 비교하였다.

`churn_any`에서는 `RandomForest`와 `DecisionTree`가 Recall과 F1 기준 더 강한 성능을 보였고, 특히 `RandomForest`는 baseline 단계에서 가장 균형 잡힌 모델로 해석되었다.

반면 `GradientBoosting`과 `XGBoost`는 Accuracy, ROC-AUC, PR-AUC는 상대적으로 높게 나타났지만, 실제 이탈자 예측을 매우 보수적으로 수행하여 Recall과 F1이 크게 낮았다.

`churn_to_mvno`에서는 희소 클래스 특성상 Accuracy보다 Recall과 PR-AUC가 더 중요했으며, 이번 실험에서는 `LogisticRegression`과 `DecisionTree`가 상대적으로 더 실질적인 baseline 역할을 수행했다.

즉 두 문제는 같은 통신사 이동 예측처럼 보이더라도 클래스 구조와 난이도가 달라, 적합한 모델 계열과 운영 기준도 다르게 해석할 필요가 있다.

---

## [ 최종 종합 비교표 ]

| 항목 | `churn_any` | `churn_to_mvno` |
|---|---|---|
| 문제 성격 | 비교적 일반적인 통신사 이동 | 매우 희소한 MVNO 이동 |
| 핵심 평가 지표 | `F1`, `ROC-AUC`, `PR-AUC`, `Recall` | `PR-AUC`, `Recall`, `Precision`, `F1` |
| 비교 모델 | LR / DT / RF / GB / XGB | LR / DT / RF / GB / XGB |
| 추가 보강 | GroupKFold, RF 튜닝, threshold 조정 | threshold 조정, PR Curve |
| 해석 포인트 | 어떤 비선형 모델 계열이 더 적합한가 | 희소 양성을 누가 더 잘 찾는가 |
| 최종 선택 기준 | 실행 결과표에서 `PR-AUC`와 `F1` 중심 | 실행 결과표에서 `PR-AUC`와 `Recall` 중심 |

---

## [ 공통 중요 변수 ]

두 문제에서 반복적으로 상위권에 나타나는 변수는 현재 데이터에서 가장 일관된 예측 신호를 제공하는 변수군으로 볼 수 있다.

특히 다음 변수들은 여러 노트북에서 반복적으로 중요하게 나타났다.

- 월평균 휴대폰 이용금액
- 통신 지출/가계 관련
- 스마트폰/5G 관련

즉, 비용/지출 관련 특성과 통신 이용 특성이 두 문제 모두에서 공통 핵심 축으로 작동하였다.

---

## [ 요약 ]

이번 보강 이후 프로젝트는
기존의 `LogisticRegression` vs `RandomForest` 중심 baseline에서 벗어나,
`DecisionTree`, `GradientBoosting`, `XGBoost`까지 포함하는 구조로 확장되었다.

또한 `churn_to_mvno`에는 threshold 분석과 PR Curve를 추가하여
희소 클래스 문제를 더 적절하게 해석할 수 있도록 보완하였다.

즉, 현재 상태에서는
- 모델 수 부족 지적 완화
- 해석 문장 최신화
- 표시용 변수명 반영
- 최종 종합 비교표 추가

까지 반영된 상태라고 정리할 수 있다.